# 04 Expert: LLM Analysis, DSS Export & Configuration

This notebook covers py2dataiku's advanced capabilities:

1. **LLM-based conversion** using `convert_with_llm()` and `MockProvider`
2. **LLM providers** -- factory pattern and provider classes
3. **LLMCodeAnalyzer** -- direct analysis with `AnalysisResult`
4. **OperationType enum** and `DataStep` internals
5. **Rule-based vs LLM-based** comparison
6. **Py2Dataiku hybrid class** with automatic fallback
7. **Configuration** via `Py2DataikuConfig` and config files
8. **DSS Export** with `DSSExporter` and `export_to_dss()`
9. **Dataset connections** and `ColumnSchema`
10. **Flow zones** for organizing large flows

## 1. LLM-Based Conversion with MockProvider

py2dataiku's recommended analysis mode uses an LLM to understand code semantics.
For demos and testing, `MockProvider` lets us work without any API key.

In [ ]:
import json
from py2dataiku import convert_with_llm, convert, MockProvider, get_provider

In [ ]:
# MockProvider returns a default JSON response for any prompt.
# This lets us exercise the full LLM pipeline without an API key.
mock = MockProvider()

# Call complete() to see the default mock response
response = mock.complete("Analyze this code")
print("Model:", response.model)
print("Content:", response.content)

In [ ]:
# MockProvider can also be initialized with custom responses.
# Keys are matched as substrings against the prompt.
custom_mock = MockProvider(responses={
    "read_csv": json.dumps({
        "code_summary": "Reads a CSV file and drops missing values",
        "total_operations": 2,
        "complexity_score": 2,
        "datasets": [
            {"name": "raw_data", "source": "data.csv", "is_input": True, "is_output": False, "inferred_columns": ["id", "name", "value"]},
            {"name": "clean_data", "source": "derived", "is_input": False, "is_output": True, "inferred_columns": ["id", "name", "value"]}
        ],
        "steps": [
            {
                "step_number": 1,
                "operation": "read_data",
                "description": "Read CSV file into DataFrame",
                "input_datasets": [],
                "output_dataset": "raw_data",
                "columns": ["id", "name", "value"],
                "suggested_recipe": "sync",
                "requires_python_recipe": False
            },
            {
                "step_number": 2,
                "operation": "drop_missing",
                "description": "Drop rows with missing values",
                "input_datasets": ["raw_data"],
                "output_dataset": "clean_data",
                "columns": [],
                "suggested_recipe": "prepare",
                "suggested_processors": ["RemoveRowsOnEmpty"],
                "requires_python_recipe": False
            }
        ],
        "recommendations": ["Consider adding data validation before dropping rows"],
        "warnings": []
    })
})

# complete_json() parses the response as JSON automatically
result = custom_mock.complete_json("Analyze: df = pd.read_csv('data.csv')")
print("Summary:", result["code_summary"])
print("Steps:", result["total_operations"])

In [ ]:
# MockProvider tracks all calls made to it (useful for testing)
print(f"Calls made to custom_mock: {len(custom_mock.calls)}")
print(f"Last prompt contained: ...{custom_mock.calls[-1]['prompt'][:50]}...")

## 2. LLM Provider Architecture

py2dataiku supports multiple LLM providers via a common `LLMProvider` ABC:

- **AnthropicProvider** -- Claude models (default: claude-sonnet-4-20250514)
- **OpenAIProvider** -- GPT models (default: gpt-4o)
- **MockProvider** -- Testing without API calls

The `get_provider()` factory creates the right provider by name.

In [ ]:
from py2dataiku import AnthropicProvider, OpenAIProvider, MockProvider, get_provider
from py2dataiku.llm.providers import LLMProvider, LLMResponse

# get_provider() is the factory function
mock_provider = get_provider("mock")
print(f"Provider type: {type(mock_provider).__name__}")
print(f"Model name: {mock_provider.model_name}")

In [ ]:
# LLMProvider defines the interface: complete(), complete_json(), model_name
# All providers return LLMResponse objects
resp = mock_provider.complete("Hello")
print(f"Response type: {type(resp).__name__}")
print(f"Fields: content={resp.content!r}, model={resp.model!r}, usage={resp.usage}")

In [ ]:
# AnthropicProvider and OpenAIProvider require API keys.
# They will raise ValueError if no key is found:
try:
    get_provider("anthropic", api_key=None)
except ValueError as e:
    print(f"Expected error: {e}")

try:
    get_provider("openai", api_key=None)
except ValueError as e:
    print(f"Expected error: {e}")

# Unknown providers also raise ValueError
try:
    get_provider("unknown_provider")
except ValueError as e:
    print(f"Expected error: {e}")

## 3. LLMCodeAnalyzer and AnalysisResult

`LLMCodeAnalyzer` is the core class that sends code to an LLM and parses the
structured response into an `AnalysisResult`.

In [ ]:
from py2dataiku import LLMCodeAnalyzer, AnalysisResult, DataStep, OperationType
from py2dataiku.llm.schemas import DatasetInfo

# Create an analyzer with a custom MockProvider that returns a rich response
mock_response = json.dumps({
    "code_summary": "ETL pipeline that reads sales data, cleans it, joins with products, and aggregates by category",
    "total_operations": 5,
    "complexity_score": 6,
    "datasets": [
        {"name": "sales", "source": "sales.csv", "is_input": True, "is_output": False, "inferred_columns": ["order_id", "product_id", "amount", "date"]},
        {"name": "products", "source": "products.csv", "is_input": True, "is_output": False, "inferred_columns": ["product_id", "name", "category"]},
        {"name": "sales_clean", "source": "derived", "is_input": False, "is_output": False, "inferred_columns": ["order_id", "product_id", "amount", "date"]},
        {"name": "sales_with_products", "source": "derived", "is_input": False, "is_output": False, "inferred_columns": ["order_id", "product_id", "amount", "date", "name", "category"]},
        {"name": "category_totals", "source": "derived", "is_input": False, "is_output": True, "inferred_columns": ["category", "total_amount", "order_count"]}
    ],
    "steps": [
        {
            "step_number": 1,
            "operation": "read_data",
            "description": "Read sales CSV file",
            "input_datasets": [],
            "output_dataset": "sales",
            "columns": ["order_id", "product_id", "amount", "date"],
            "suggested_recipe": "sync",
            "requires_python_recipe": False,
            "reasoning": "Simple file read maps to sync recipe"
        },
        {
            "step_number": 2,
            "operation": "read_data",
            "description": "Read products CSV file",
            "input_datasets": [],
            "output_dataset": "products",
            "columns": ["product_id", "name", "category"],
            "suggested_recipe": "sync",
            "requires_python_recipe": False,
            "reasoning": "Simple file read maps to sync recipe"
        },
        {
            "step_number": 3,
            "operation": "drop_missing",
            "description": "Remove rows with null amounts",
            "input_datasets": ["sales"],
            "output_dataset": "sales_clean",
            "columns": ["amount"],
            "suggested_recipe": "prepare",
            "suggested_processors": ["RemoveRowsOnEmpty"],
            "requires_python_recipe": False,
            "reasoning": "dropna maps to RemoveRowsOnEmpty processor in prepare recipe"
        },
        {
            "step_number": 4,
            "operation": "join",
            "description": "Join sales with products on product_id",
            "input_datasets": ["sales_clean", "products"],
            "output_dataset": "sales_with_products",
            "columns": ["product_id"],
            "join_conditions": [{"left_column": "product_id", "right_column": "product_id", "operator": "equals"}],
            "join_type": "left",
            "suggested_recipe": "join",
            "requires_python_recipe": False,
            "reasoning": "pd.merge with on='product_id' maps to join recipe"
        },
        {
            "step_number": 5,
            "operation": "group_aggregate",
            "description": "Aggregate sales by category",
            "input_datasets": ["sales_with_products"],
            "output_dataset": "category_totals",
            "columns": ["category", "amount"],
            "group_by_columns": ["category"],
            "aggregations": [
                {"column": "amount", "function": "sum", "output_column": "total_amount"},
                {"column": "order_id", "function": "count", "output_column": "order_count"}
            ],
            "suggested_recipe": "grouping",
            "requires_python_recipe": False,
            "reasoning": "groupby().agg() maps to grouping recipe"
        }
    ],
    "recommendations": [
        "Consider filtering before join to reduce data volume",
        "Add data quality checks on the amount column"
    ],
    "warnings": ["Date column not parsed -- consider adding date parsing step"]
})

# Create analyzer with the custom mock
mock_for_analysis = MockProvider(responses={"Analyze": mock_response})
analyzer = LLMCodeAnalyzer(provider=mock_for_analysis)
print(f"Analyzer provider: {analyzer.provider.model_name}")

In [ ]:
# Analyze code -- the LLM (mock) extracts data manipulation steps
code = """
import pandas as pd
sales = pd.read_csv('sales.csv')
products = pd.read_csv('products.csv')
sales_clean = sales.dropna(subset=['amount'])
merged = sales_clean.merge(products, on='product_id', how='left')
result = merged.groupby('category').agg(
    total_amount=('amount', 'sum'),
    order_count=('order_id', 'count')
)
"""

analysis = analyzer.analyze(code)
print(f"Type: {type(analysis).__name__}")
print(f"Code summary: {analysis.code_summary}")
print(f"Complexity score: {analysis.complexity_score}/10")
print(f"Total operations: {analysis.total_operations}")
print(f"Model used: {analysis.model_used}")

In [ ]:
# Examine datasets identified by the LLM
print(f"Datasets found: {len(analysis.datasets)}")
for ds in analysis.datasets:
    role = "INPUT" if ds.is_input else ("OUTPUT" if ds.is_output else "INTERMEDIATE")
    print(f"  {ds.name}: {role} (source={ds.source}, columns={ds.inferred_columns})")

In [ ]:
# Examine each analysis step
print(f"Steps: {len(analysis.steps)}\n")
for step in analysis.steps:
    print(f"Step {step.step_number}: {step.description}")
    print(f"  Operation: {step.operation.value}")
    print(f"  Inputs: {step.input_datasets} -> Output: {step.output_dataset}")
    print(f"  Suggested recipe: {step.suggested_recipe}")
    if step.suggested_processors:
        print(f"  Suggested processors: {step.suggested_processors}")
    if step.reasoning:
        print(f"  Reasoning: {step.reasoning}")
    print()

In [ ]:
# AnalysisResult also includes recommendations and warnings
print("Recommendations:")
for rec in analysis.recommendations:
    print(f"  - {rec}")

print("\nWarnings:")
for warn in analysis.warnings:
    print(f"  - {warn}")

In [ ]:
# AnalysisResult supports serialization
analysis_dict = analysis.to_dict()
print("Keys:", list(analysis_dict.keys()))

# Convert to JSON string
analysis_json = analysis.to_json(indent=2)
print(f"\nJSON length: {len(analysis_json)} chars")
print(analysis_json[:200], "...")

In [ ]:
# Round-trip: reconstruct from dict
restored = AnalysisResult.from_dict(analysis_dict)
print(f"Restored summary: {restored.code_summary}")
print(f"Restored steps: {len(restored.steps)}")
print(f"Restored complexity: {restored.complexity_score}")

## 4. OperationType Enum and DataStep Details

`OperationType` defines all 28 operation types the LLM can detect.
`DataStep` holds the full details for each operation.

In [ ]:
from py2dataiku import OperationType

# List all operation types grouped by category
categories = {
    "Data I/O": [OperationType.READ_DATA, OperationType.WRITE_DATA],
    "Transformations": [
        OperationType.FILTER, OperationType.SELECT_COLUMNS,
        OperationType.DROP_COLUMNS, OperationType.RENAME_COLUMNS,
        OperationType.ADD_COLUMN, OperationType.TRANSFORM_COLUMN,
    ],
    "Missing Values": [OperationType.FILL_MISSING, OperationType.DROP_MISSING],
    "Deduplication": [OperationType.DROP_DUPLICATES],
    "Aggregation": [OperationType.GROUP_AGGREGATE, OperationType.WINDOW_FUNCTION],
    "Combining": [OperationType.JOIN, OperationType.UNION],
    "Reshaping": [OperationType.PIVOT, OperationType.UNPIVOT],
    "Sorting": [OperationType.SORT, OperationType.TOP_N, OperationType.SAMPLE],
    "Type Conversion": [OperationType.CAST_TYPE, OperationType.PARSE_DATE],
    "Column Ops": [OperationType.SPLIT_COLUMN, OperationType.ENCODE_CATEGORICAL],
    "Scaling": [OperationType.NORMALIZE_SCALE],
    "Geographic": [OperationType.GEO_OPERATION],
    "Custom": [OperationType.CUSTOM_FUNCTION, OperationType.UNKNOWN],
}

total = 0
for cat, ops in categories.items():
    print(f"{cat}:")
    for op in ops:
        print(f"  {op.value}")
        total += 1
print(f"\nTotal: {total} operation types")

In [ ]:
# DataStep holds operation-specific details.
# Let's examine the join step from our analysis.
join_step = analysis.steps[3]  # Step 4: join
print(f"Operation: {join_step.operation.value}")
print(f"Join type: {join_step.join_type}")
print(f"Join conditions:")
for jc in join_step.join_conditions:
    print(f"  {jc.left_column} {jc.operator} {jc.right_column}")

# And the aggregation step
agg_step = analysis.steps[4]  # Step 5: group_aggregate
print(f"\nGroup by: {agg_step.group_by_columns}")
print(f"Aggregations:")
for agg in agg_step.aggregations:
    print(f"  {agg.function}({agg.column}) -> {agg.output_column}")

In [ ]:
# DataStep.to_dict() for serialization
step_dict = join_step.to_dict()
print(json.dumps(step_dict, indent=2))

## 5. Optimization Suggestions from LLMCodeAnalyzer

The analyzer can provide optimization suggestions based on the analysis result.

In [ ]:
# get_optimization_suggestions examines the analysis for improvement opportunities
suggestions = analyzer.get_optimization_suggestions(analysis)
print(f"Optimization suggestions: {len(suggestions)}")
for s in suggestions:
    print(f"  - {s}")

## 6. Rule-Based vs LLM-Based Comparison

py2dataiku offers two analysis modes. Let's compare them on the same code.

In [ ]:
from py2dataiku import convert, LLMFlowGenerator

comparison_code = """
import pandas as pd
df = pd.read_csv('data.csv')
df = df.dropna()
df = df.rename(columns={'old_name': 'new_name'})
result = df.groupby('category').agg({'amount': 'sum'})
result.to_csv('output.csv')
"""

# Rule-based conversion (uses AST pattern matching)
rule_flow = convert(comparison_code)
print("=== Rule-Based Flow ===")
print(f"Recipes: {len(rule_flow.recipes)}")
for r in rule_flow.recipes:
    print(f"  {r.name}: {r.recipe_type.value} ({r.inputs} -> {r.outputs})")
print(f"Datasets: {len(rule_flow.datasets)}")

In [ ]:
# LLM-based conversion (using our MockProvider with a tailored response)
llm_mock_response = json.dumps({
    "code_summary": "Reads CSV, cleans, renames, aggregates by category, writes output",
    "total_operations": 5,
    "complexity_score": 4,
    "datasets": [
        {"name": "data", "source": "data.csv", "is_input": True, "is_output": False, "inferred_columns": ["old_name", "category", "amount"]},
        {"name": "data_clean", "source": "derived", "is_input": False, "is_output": False, "inferred_columns": ["old_name", "category", "amount"]},
        {"name": "data_renamed", "source": "derived", "is_input": False, "is_output": False, "inferred_columns": ["new_name", "category", "amount"]},
        {"name": "category_summary", "source": "derived", "is_input": False, "is_output": True, "inferred_columns": ["category", "amount"]}
    ],
    "steps": [
        {"step_number": 1, "operation": "read_data", "description": "Read CSV", "input_datasets": [], "output_dataset": "data", "columns": [], "suggested_recipe": "sync", "requires_python_recipe": False},
        {"step_number": 2, "operation": "drop_missing", "description": "Drop nulls", "input_datasets": ["data"], "output_dataset": "data_clean", "columns": [], "suggested_recipe": "prepare", "suggested_processors": ["RemoveRowsOnEmpty"], "requires_python_recipe": False},
        {"step_number": 3, "operation": "rename_columns", "description": "Rename old_name to new_name", "input_datasets": ["data_clean"], "output_dataset": "data_renamed", "columns": ["old_name"], "rename_mapping": {"old_name": "new_name"}, "suggested_recipe": "prepare", "suggested_processors": ["ColumnRenamer"], "requires_python_recipe": False},
        {"step_number": 4, "operation": "group_aggregate", "description": "Sum amount by category", "input_datasets": ["data_renamed"], "output_dataset": "category_summary", "columns": ["category", "amount"], "group_by_columns": ["category"], "aggregations": [{"column": "amount", "function": "sum", "output_column": "amount"}], "suggested_recipe": "grouping", "requires_python_recipe": False},
        {"step_number": 5, "operation": "write_data", "description": "Write to CSV", "input_datasets": ["category_summary"], "output_dataset": "output", "columns": [], "suggested_recipe": "sync", "requires_python_recipe": False}
    ],
    "recommendations": ["Consecutive prepare steps (drop_missing + rename) could be merged into one Prepare recipe"],
    "warnings": []
})

llm_mock = MockProvider(responses={"Analyze": llm_mock_response})
llm_analyzer = LLMCodeAnalyzer(provider=llm_mock)
llm_analysis = llm_analyzer.analyze(comparison_code)

# Generate flow from LLM analysis
llm_gen = LLMFlowGenerator()
llm_flow = llm_gen.generate(llm_analysis, flow_name="llm_pipeline")

print("=== LLM-Based Flow ===")
print(f"Recipes: {len(llm_flow.recipes)}")
for r in llm_flow.recipes:
    print(f"  {r.name}: {r.recipe_type.value} ({r.inputs} -> {r.outputs})")
print(f"Datasets: {len(llm_flow.datasets)}")

In [ ]:
# Key differences between the two approaches:
print("Comparison Summary:")
print(f"  Rule-based: {len(rule_flow.recipes)} recipes, {len(rule_flow.datasets)} datasets")
print(f"  LLM-based:  {len(llm_flow.recipes)} recipes, {len(llm_flow.datasets)} datasets")
print()
print("Rule-based advantages:")
print("  - Fast, deterministic, offline")
print("  - No API key needed")
print("  - Predictable output")
print()
print("LLM-based advantages:")
print("  - Understands code semantics")
print("  - Handles complex/ambiguous patterns")
print("  - Provides reasoning and recommendations")
print("  - Better accuracy for non-standard code")

## 7. Py2Dataiku Hybrid Class

The `Py2Dataiku` class provides a unified interface with automatic fallback:
if LLM initialization fails, it falls back to rule-based analysis.

In [ ]:
from py2dataiku import Py2Dataiku

# Without a valid API key, Py2Dataiku falls back to rule-based
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    converter = Py2Dataiku(provider="anthropic", use_llm=True)
    if w:
        print(f"Warning: {w[0].message}")
    print(f"Using LLM: {converter.use_llm}")

In [ ]:
# Rule-based mode works without any API key
converter_rules = Py2Dataiku(use_llm=False)
print(f"Using LLM: {converter_rules.use_llm}")

simple_code = """
import pandas as pd
df = pd.read_csv('input.csv')
df['total'] = df['price'] * df['qty']
df.to_csv('output.csv')
"""

flow = converter_rules.convert(simple_code, flow_name="hybrid_demo")
print(f"\nFlow: {flow.name}")
print(f"Recipes: {len(flow.recipes)}")
for r in flow.recipes:
    print(f"  {r.name}: {r.recipe_type.value}")

In [ ]:
# Py2Dataiku also wraps visualization methods
ascii_viz = converter_rules.visualize(flow, format="ascii")
print(ascii_viz)

In [ ]:
# The analyze() method is only available in LLM mode
try:
    converter_rules.analyze(simple_code)
except ValueError as e:
    print(f"Expected error: {e}")

## 8. Configuration with Py2DataikuConfig

`Py2DataikuConfig` centralizes all settings. Configuration can come from:
- Python code (direct construction)
- TOML files (`py2dataiku.toml`)
- YAML files (`.py2dataiku.yaml` / `.py2dataiku.yml`)
- RC files (`.py2dataikurc`)
- Environment variables (`PY2DATAIKU_PROVIDER`, `PY2DATAIKU_PROJECT_KEY`)

In [ ]:
from py2dataiku import Py2DataikuConfig, load_config, find_config_file

# Default configuration
config = Py2DataikuConfig()
print("Default configuration:")
print(f"  Provider: {config.default_provider}")
print(f"  Model: {config.default_model}")
print(f"  Project key: {config.project_key}")
print(f"  Flow name: {config.flow_name}")
print(f"  Optimize: {config.optimize}")
print(f"  Optimization level: {config.optimization_level}")
print(f"  Default format: {config.default_format}")
print(f"  Default connection: {config.default_connection}")

In [ ]:
# Custom configuration
custom_config = Py2DataikuConfig(
    default_provider="openai",
    default_model="gpt-4o",
    project_key="SALES_PIPELINE",
    flow_name="sales_etl",
    optimize=True,
    optimization_level=2,
    dataset_prefix="src_",
    dataset_suffix="_v2",
    recipe_prefix="compute_",
    default_format="html",
    default_connection="PostgreSQL",
)

print("Custom configuration:")
print(f"  Provider: {custom_config.default_provider}")
print(f"  Project key: {custom_config.project_key}")
print(f"  Dataset prefix: {custom_config.dataset_prefix!r}")
print(f"  Dataset suffix: {custom_config.dataset_suffix!r}")
print(f"  Recipe prefix: {custom_config.recipe_prefix!r}")
print(f"  Connection: {custom_config.default_connection}")

In [ ]:
# Config can be serialized to dict and back
config_dict = custom_config.to_dict()
print("Serialized config:")
print(json.dumps(config_dict, indent=2))

In [ ]:
# Round-trip via from_dict
restored_config = Py2DataikuConfig.from_dict(config_dict)
print(f"Restored provider: {restored_config.default_provider}")
print(f"Restored project key: {restored_config.project_key}")
print(f"Restored connection: {restored_config.default_connection}")

In [ ]:
# Config file discovery: find_config_file() searches current dir, then home
# Supported filenames (in order): py2dataiku.toml, .py2dataikurc, .py2dataiku.yaml, .py2dataiku.yml
from py2dataiku.config import CONFIG_FILE_NAMES
print("Config file search order:")
for name in CONFIG_FILE_NAMES:
    print(f"  {name}")

# load_config() auto-discovers and loads config files
# It also supports environment variable overrides:
#   PY2DATAIKU_PROVIDER -> config.default_provider
#   PY2DATAIKU_PROJECT_KEY -> config.project_key
config_from_env = load_config(auto_discover=False)  # Defaults only
print(f"\nLoaded config (defaults): {config_from_env.project_key}")

In [ ]:
# Example: what a TOML config file would look like
toml_example = """
# py2dataiku.toml
[provider]
default = "anthropic"
model = "claude-sonnet-4-20250514"

[project]
key = "SALES_PIPELINE"
flow_name = "sales_etl"

[optimization]
enabled = true
level = 2

[naming]
dataset_prefix = "src_"
dataset_suffix = ""
recipe_prefix = "compute_"
recipe_suffix = ""

[output]
format = "svg"
connection = "PostgreSQL"
"""
print(toml_example)

# YAML equivalent
yaml_example = """
# .py2dataiku.yaml
provider:
  default: anthropic
  model: claude-sonnet-4-20250514
project:
  key: SALES_PIPELINE
  flow_name: sales_etl
optimization:
  enabled: true
  level: 2
"""
print(yaml_example)

## 9. DSS Export with DSSExporter

`DSSExporter` generates a complete Dataiku DSS project structure that can
be imported directly into a DSS instance.

In [ ]:
from py2dataiku import DSSExporter, DSSProjectConfig, export_to_dss

# Create a flow to export
export_code = """
import pandas as pd
df = pd.read_csv('transactions.csv')
df = df.dropna(subset=['amount'])
df['amount'] = df['amount'].round(2)
summary = df.groupby('category').agg({'amount': 'sum'})
"""
export_flow = convert(export_code)
print(f"Flow to export: {len(export_flow.recipes)} recipes, {len(export_flow.datasets)} datasets")

In [ ]:
# DSSProjectConfig holds project-level settings
project_config = DSSProjectConfig(
    project_key="TRANSACTION_ETL",
    project_name="Transaction Processing Pipeline",
    owner="data_team",
    description="Auto-generated from Python ETL code",
    tags=["py2dataiku", "transactions", "etl"],
    default_connection="filesystem_managed",
    default_format="csv",
    include_code_comments=True,
)

print("Project config:")
print(json.dumps(project_config.to_dict(), indent=2))

In [ ]:
# DSSExporter takes a flow and config
exporter = DSSExporter(flow=export_flow, config=project_config)

# get_api_bundle() returns the full project as a dictionary
# (useful for API-based import without writing to disk)
bundle = exporter.get_api_bundle()
print(f"Bundle keys: {list(bundle.keys())}")
print(f"Project key: {bundle['projectKey']}")
print(f"Datasets in bundle: {len(bundle['datasets'])}")
print(f"Recipes in bundle: {len(bundle['recipes'])}")

In [ ]:
# Export to disk (creates full DSS project directory structure)
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    export_path = exporter.export(tmpdir)
    
    # Show the directory structure
    print(f"Exported to: {export_path}")
    print("\nProject structure:")
    for root, dirs, files in os.walk(export_path):
        level = root.replace(export_path, "").count(os.sep)
        indent = "  " * level
        basename = os.path.basename(root) or os.path.basename(export_path)
        print(f"{indent}{basename}/")
        sub_indent = "  " * (level + 1)
        for file in sorted(files):
            print(f"{sub_indent}{file}")

In [ ]:
# Convenience function: export_to_dss()
with tempfile.TemporaryDirectory() as tmpdir:
    path = export_to_dss(
        flow=export_flow,
        output_dir=tmpdir,
        project_key="QUICK_EXPORT",
        create_zip=False,
    )
    print(f"Exported to: {path}")
    # List files
    for item in sorted(os.listdir(path)):
        print(f"  {item}")

In [ ]:
# Export as zip for easy import into DSS
with tempfile.TemporaryDirectory() as tmpdir:
    zip_path = export_to_dss(
        flow=export_flow,
        output_dir=tmpdir,
        project_key="ZIP_EXPORT",
        create_zip=True,
    )
    print(f"Zip file: {os.path.basename(zip_path)}")
    print(f"Zip size: {os.path.getsize(zip_path)} bytes")

## 10. DatasetConnectionType and ColumnSchema

Datasets can be configured with specific connection types (13 supported)
and column schemas.

In [ ]:
from py2dataiku import DatasetConnectionType, ColumnSchema, DataikuDataset, DatasetType

# All 13 connection types
print("Supported connection types:")
for ct in DatasetConnectionType:
    print(f"  {ct.name}: {ct.value}")

In [ ]:
# ColumnSchema defines typed columns with nullability and defaults
schema = [
    ColumnSchema(name="user_id", type="int", nullable=False),
    ColumnSchema(name="username", type="string", nullable=False),
    ColumnSchema(name="email", type="string", nullable=True),
    ColumnSchema(name="created_at", type="date", nullable=False, format="yyyy-MM-dd"),
    ColumnSchema(name="is_active", type="boolean", nullable=False, default=True),
    ColumnSchema(name="balance", type="float", nullable=True, default=0.0),
]

print("Schema columns:")
for col in schema:
    extras = []
    if not col.nullable:
        extras.append("NOT NULL")
    if col.default is not None:
        extras.append(f"default={col.default}")
    if col.format:
        extras.append(f"format={col.format}")
    extra_str = f" ({', '.join(extras)})" if extras else ""
    print(f"  {col.name}: {col.type}{extra_str}")

In [ ]:
# Create a dataset with a specific connection type and schema
ds = DataikuDataset(
    name="users_table",
    dataset_type=DatasetType.INPUT,
    connection_type=DatasetConnectionType.SQL_POSTGRESQL,
    schema=schema,
)

print(f"Dataset: {ds.name}")
print(f"Type: {ds.dataset_type.value}")
print(f"Connection: {ds.connection_type.value}")
print(f"Is input: {ds.is_input}")
print(f"Schema columns: {len(ds.schema)}")

In [ ]:
# add_column() is a convenience method for building schemas
ds2 = DataikuDataset(
    name="events",
    dataset_type=DatasetType.INTERMEDIATE,
    connection_type=DatasetConnectionType.S3,
)
ds2.add_column("event_id", "string", nullable=False)
ds2.add_column("timestamp", "date")
ds2.add_column("payload", "string")
ds2.add_note("Events from Kafka ingestion pipeline")

print(f"Dataset: {ds2.name} ({ds2.connection_type.value})")
print(f"Columns: {[c.name for c in ds2.schema]}")
print(f"Notes: {ds2.notes}")

In [ ]:
# Serialization: to_dict() and to_json() for API compatibility
ds_dict = ds.to_dict()
print("to_dict():")
print(json.dumps(ds_dict, indent=2))

In [ ]:
# to_json() produces Dataiku API-compatible format
ds_json = ds.to_json()
print("to_json() (API-compatible):")
print(json.dumps(ds_json, indent=2))

## 11. Recipe Configs and to_recipe_configs()

`DataikuFlow.to_recipe_configs()` produces Dataiku API-compatible recipe
configuration objects.

In [ ]:
# to_recipe_configs() returns API-compatible recipe dictionaries
recipe_configs = export_flow.to_recipe_configs()
print(f"Recipe configs: {len(recipe_configs)}")
for rc in recipe_configs:
    print(f"\n  Name: {rc.get('name', 'N/A')}")
    print(f"  Type: {rc.get('type', 'N/A')}")
    print(f"  Inputs: {rc.get('inputs', [])}")
    print(f"  Outputs: {rc.get('outputs', [])}")

## 12. Flow Zones

`FlowZone` provides logical grouping of datasets and recipes within a flow.
Zones help organize large flows in the Dataiku DSS UI.

In [ ]:
from py2dataiku import FlowZone, DataikuFlow

# Create a flow with zones
flow = convert("""
import pandas as pd
df = pd.read_csv('raw.csv')
df = df.dropna()
df = df.rename(columns={'col1': 'feature1'})
result = df.groupby('group').agg({'feature1': 'mean'})
""")

print(f"Flow has {len(flow.recipes)} recipes and {len(flow.datasets)} datasets")
print(f"Datasets: {[ds.name for ds in flow.datasets]}")
print(f"Recipes: {[r.name for r in flow.recipes]}")

In [ ]:
# Create zones to organize the flow
ingestion_zone = FlowZone(
    name="Data Ingestion",
    color="#2ecc71",  # Green
)

processing_zone = FlowZone(
    name="Data Processing",
    color="#3498db",  # Blue
)

analytics_zone = FlowZone(
    name="Analytics",
    color="#e74c3c",  # Red
)

# Add datasets and recipes to zones
# (use dataset/recipe names from the flow)
for ds in flow.datasets:
    if ds.dataset_type == DatasetType.INPUT:
        ingestion_zone.add_dataset(ds.name)
    elif ds.dataset_type == DatasetType.OUTPUT:
        analytics_zone.add_dataset(ds.name)
    else:
        processing_zone.add_dataset(ds.name)

for r in flow.recipes:
    if r.recipe_type.value in ("grouping", "window", "pivot"):
        analytics_zone.add_recipe(r.name)
    else:
        processing_zone.add_recipe(r.name)

print(f"Ingestion zone: {ingestion_zone.datasets}")
print(f"Processing zone: {processing_zone.datasets}, recipes={processing_zone.recipes}")
print(f"Analytics zone: {analytics_zone.datasets}, recipes={analytics_zone.recipes}")

In [ ]:
# Add zones to the flow
flow.add_zone(ingestion_zone)
flow.add_zone(processing_zone)
flow.add_zone(analytics_zone)

print(f"Flow now has {len(flow.zones)} zones")

# Retrieve a zone by name
retrieved = flow.get_zone("Data Processing")
print(f"Retrieved zone: {retrieved.name} (color={retrieved.color})")

In [ ]:
# Zones are included in serialization
flow_dict = flow.to_dict()
print(f"Flow dict has 'zones' key: {'zones' in flow_dict}")
print(f"Zones in dict: {len(flow_dict['zones'])}")
for z in flow_dict["zones"]:
    print(f"  {z['name']}: {len(z['datasets'])} datasets, {len(z['recipes'])} recipes")

In [ ]:
# Zones survive round-trip serialization
flow_json = flow.to_json()
restored_flow = DataikuFlow.from_json(flow_json)
print(f"Restored flow zones: {len(restored_flow.zones)}")
for z in restored_flow.zones:
    print(f"  {z.name}: color={z.color}, datasets={z.datasets}, recipes={z.recipes}")

## 13. Putting It All Together: End-to-End Expert Workflow

Combine LLM analysis, configuration, zones, and DSS export in a single workflow.

In [ ]:
# Step 1: Configure the project
project_cfg = Py2DataikuConfig(
    project_key="E2E_DEMO",
    flow_name="end_to_end_pipeline",
    optimize=True,
    optimization_level=2,
    default_connection="PostgreSQL",
)
print(f"Project: {project_cfg.project_key}")

# Step 2: Convert code (rule-based for this demo)
pipeline_code = """
import pandas as pd
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
orders = orders.dropna(subset=['total'])
orders['total'] = orders['total'].round(2)
merged = orders.merge(customers, on='customer_id', how='left')
by_region = merged.groupby('region').agg({'total': 'sum'})
"""
e2e_flow = convert(pipeline_code)
e2e_flow.name = project_cfg.flow_name
print(f"\nFlow: {e2e_flow.name}")
print(f"Recipes: {len(e2e_flow.recipes)}, Datasets: {len(e2e_flow.datasets)}")

In [ ]:
# Step 3: Organize into zones
src_zone = FlowZone(name="Sources", color="#27ae60")
transform_zone = FlowZone(name="Transformations", color="#2980b9")
output_zone = FlowZone(name="Outputs", color="#8e44ad")

for ds in e2e_flow.datasets:
    if ds.dataset_type == DatasetType.INPUT:
        src_zone.add_dataset(ds.name)
    elif ds.dataset_type == DatasetType.OUTPUT:
        output_zone.add_dataset(ds.name)
    else:
        transform_zone.add_dataset(ds.name)

for r in e2e_flow.recipes:
    transform_zone.add_recipe(r.name)

e2e_flow.add_zone(src_zone)
e2e_flow.add_zone(transform_zone)
e2e_flow.add_zone(output_zone)

print("Zones configured:")
for z in e2e_flow.zones:
    print(f"  {z.name}: {len(z.datasets)} datasets, {len(z.recipes)} recipes")

In [ ]:
# Step 4: Export to DSS format
dss_config = DSSProjectConfig(
    project_key=project_cfg.project_key,
    project_name="End-to-End Demo Pipeline",
    owner="data_team",
    tags=["demo", "e2e", "py2dataiku"],
)

exporter = DSSExporter(flow=e2e_flow, config=dss_config)
bundle = exporter.get_api_bundle()
print(f"API bundle ready: {bundle['projectKey']}")
print(f"  {len(bundle['datasets'])} datasets")
print(f"  {len(bundle['recipes'])} recipes")
print(f"  Generated: {bundle['metadata']['timestamp'][:19]}")

In [ ]:
# Step 5: Visualize the final flow
print(e2e_flow.get_summary())

In [ ]:
# Validation check
validation = e2e_flow.validate()
print(f"Valid: {validation['valid']}")
if validation['errors']:
    print(f"Errors: {validation['errors']}")
if validation['warnings']:
    print(f"Warnings: {len(validation['warnings'])}")
if validation['info']:
    print(f"Info: {len(validation['info'])}")

## Summary

This notebook covered py2dataiku's expert-level features:

| Feature | Key Classes/Functions |
|---|---|
| LLM Providers | `MockProvider`, `AnthropicProvider`, `OpenAIProvider`, `get_provider()` |
| LLM Analysis | `LLMCodeAnalyzer.analyze()` returns `AnalysisResult` |
| Analysis Schema | `AnalysisResult`, `DataStep`, `OperationType` (28 types), `DatasetInfo` |
| Hybrid Conversion | `Py2Dataiku` class with LLM-first, rule-based fallback |
| Configuration | `Py2DataikuConfig`, `load_config()`, TOML/YAML/RC/env support |
| DSS Export | `DSSExporter.export()`, `export_to_dss()`, `DSSProjectConfig` |
| Dataset Connections | `DatasetConnectionType` (13 types), `ColumnSchema` |
| API Output | `flow.to_recipe_configs()` for Dataiku API-compatible configs |
| Flow Zones | `FlowZone` for organizing datasets and recipes |

**Next**: See `05_master.ipynb` for extensibility, scenarios, MLOps, and pipeline examples.